<a href="https://colab.research.google.com/github/Vysokodelovoi/IAD_GP/blob/main/eda_univariate_cat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [227]:
# !pip install country_converter --upgrade

In [228]:
# !pip install pycountry

In [229]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio
import plotly.graph_objects as go
import country_converter as coco
import pycountry
import gettext

In [230]:
df = pd.read_parquet('main_df.parquet')

In [231]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118898 entries, 0 to 118897
Data columns (total 30 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   hotel                           118898 non-null  category      
 1   is_canceled                     118898 non-null  int64         
 2   lead_time                       118898 non-null  int64         
 3   arrival_date_year               118898 non-null  int64         
 4   arrival_date_month              118898 non-null  category      
 5   arrival_date_week_number        118898 non-null  int64         
 6   arrival_date_day_of_month       118898 non-null  int64         
 7   stays_in_weekend_nights         118898 non-null  int64         
 8   stays_in_week_nights            118898 non-null  int64         
 9   adults                          118898 non-null  int64         
 10  children                        118898 non-null  int64  

In [232]:
custom_style = go.layout.Template(pio.templates["plotly_white"])

custom_style.layout.update(
    paper_bgcolor="#fcfbfa",
    plot_bgcolor="#f7f5f0",
    font=dict(
        family="Segoe UI, Arial, sans-serif",
        size=13,
        color="#3d4047"
    ),

    margin=dict(t=80, b=60, l=60, r=40),
    title=dict(
        font=dict(size=22, color="#1c212d", weight="bold"),
        pad=dict(b=10)
    )
)

custom_style.layout.colorway = [
    "#deb887",
    "#e28743",
    "#6b8e6b",
    "#e06d75",
    "#205072"
]

axis_config = dict(
    gridcolor="#eae6df",
    zerolinecolor="#d3cbd0",
    tickcolor="#c8becc",
    ticklen=6,
    linecolor="#eae6df",
    showgrid=True,
    ticks="outside",
    title=dict(font=dict(weight="bold", color="#1c212d")),
    tickfont=dict(
        size=12,
        color="#545861",
        family="Arial, sans-serif"
    )
)

custom_style.layout.xaxis = go.layout.XAxis(axis_config)
custom_style.layout.yaxis = go.layout.YAxis(axis_config)

custom_style.layout.legend = dict(
    bgcolor="rgba(252, 251, 250, 0.9)",
    bordercolor="#eae6df",
    borderwidth=1,
    orientation="h",
    yanchor="bottom", y=1.02,
    xanchor="right", x=1
)

custom_style.data.pie = [
    go.Pie(
        textposition="outside",
        textinfo="percent+label",
        textfont=dict(
            size=14,
            color="#2a2c30",
            family="Arial, sans-serif"
        )
    )
]

custom_style.layout.geo = go.layout.Geo(
    showframe=False,
    showcoastlines=True,
    coastlinecolor="#eae6df",
    projection_type="equirectangular",
    landcolor="#f7f5f0",
    lakecolor="#fcfbfa",
    bgcolor="#fcfbfa"
)

pio.templates["summer_minimal"] = custom_style
pio.templates.default = "summer_minimal"

In [233]:
df_no_dup = df.drop_duplicates()

In [234]:
len(df_no_dup), len(df)

(86914, 118898)

In [235]:
from google.colab import drive

In [236]:
drive.mount('/content/drive')
path = '/content/drive/MyDrive/gp_iad'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Univariate analysis категориальных переменных (с дубликатами)

## `hotel`

In [237]:
fig = px.histogram(df,
             x='hotel',
             title='Типы отелей в датасете')

fig.update_layout(
    xaxis_title='тип отеля',
    yaxis_title='количество'
)
fig.show()

In [238]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

## `meal`

In [239]:
meals = df.groupby('meal')['hotel'].count().reset_index()

/tmp/ipykernel_2167/158431835.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [240]:
meals

,meal,hotel
0,BB,91863
1,FB,798
2,HB,14434
3,SC,10638
4,Undefined,1165


В столбце есть значения 'undefined'. Избавимся от них, обновив категории.

In [241]:
df = df[df.meal != 'Undefined'].copy()

In [242]:
df['meal'] = df['meal'].astype('category')

In [243]:
df['meal'] = df['meal'].cat.remove_unused_categories()

In [244]:
meals = df.groupby('meal')['hotel'].count().reset_index()

/tmp/ipykernel_2167/158431835.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [245]:
meals

,meal,hotel
0,BB,91863
1,FB,798
2,HB,14434
3,SC,10638


In [246]:
fig = px.pie(meals,
             values='hotel',
             names='meal',
             title='Типы пансионов',
             labels={
                'BB': 'Завтрак',
                'HB': 'Полупансион',
                'FB': 'Полный пансион',
                'SC': 'Без питания'
            })

leg = {
    'BB': 'Завтрак',
    'HB': 'Полупансион',
    'FB': 'Полный пансион',
    'SC': 'Без питания'
}

fig.for_each_trace(lambda t: t.update(labels=[leg.get(x, x) for x in t.labels]))

fig.show()

Интересно, что в питание у большинства бронирующих был включен только завтрак, вовсе без питания тариф выбрали практически 10 процентов.

In [247]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

## `country`

In [248]:
df.country.unique()

['PRT', 'GBR', 'USA', 'ESP', 'IRL', ..., 'KIR', 'SDN', 'ATF', 'SLE', 'LAO']
Length: 177
Categories (177, object): ['ABW', 'AGO', 'AIA', 'ALB', ..., 'VNM', 'ZAF', 'ZMB', 'ZWE']

Преобразуем коды стран в полное название, а также переведем названия стран на русский для более удобного восприятия.

In [249]:
import pycountry
import gettext

In [250]:
def get_country_name(code):
    try:
        country = pycountry.countries.get(alpha_3=code)
        return country.name
    except:
        return code

In [251]:
df['country_full'] = df['country'].astype(str).apply(get_country_name)
df['country_full'] = df['country_full'].astype('category')

In [252]:
countries = df.groupby('country_full')['country'].count().reset_index()

/tmp/ipykernel_2167/1566493600.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [253]:
countries

,country_full,country
0,Albania,12
1,Algeria,103
2,American Samoa,1
3,Andorra,7
4,Angola,362
...,...,...
172,"Venezuela, Bolivarian Republic of",26
173,Viet Nam,8
174,"Virgin Islands, British",1
175,Zambia,2


In [254]:
countries = countries.sort_values(by='country', ascending=False)

In [255]:
countries

,country_full,country
130,Portugal,47887
167,United Kingdom,12109
55,France,10324
149,Spain,8271
60,Germany,7286
...,...,...
116,New Caledonia,1
118,Nicaragua,1
151,Sudan,1
169,United States Minor Outlying Islands,1


In [256]:
countries['log_count'] = np.log(countries['country'])

In [257]:
fig = px.choropleth(
    countries,
    locations="country_full",
    locationmode="country names",
    color="log_count",
    color_continuous_scale = "Tealrose",
    title="География гостей в отелях (логнормированная)",
    hover_data={
        "log_count": False,
        "country_full": True,
        "country": True
    },
    labels={
        "country_full": "Страна",
        "country": "Количество гостей",
        "log_count": "логарифм количества гостей"
    }
)

fig.show()

Больше всего броней исходило от жителей стран Западной Европы, а также Великобритании, США, Бразилии. Меньше всего - от жителей из стран Африки, Юго-Восточной Азии и Океании.

In [258]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

## `arrival_date_month`

Чтобы правильно отсортировать данные на графике, добавлю столбец с номером месяца.

In [259]:
mlist = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
df['arrival_date_month_num'] = df['arrival_date_month'].apply(lambda x: mlist.index(x))

In [260]:
mon = df.groupby(['arrival_date_month_num', 'arrival_date_month'], observed=True)['hotel'].count().reset_index()

In [261]:
mon['arrival_date_month_num'] = mon['arrival_date_month_num'].astype(int)
mon = mon.sort_values('arrival_date_month_num')

In [262]:
mon

,arrival_date_month_num,arrival_date_month,hotel
4,0,January,5746
3,1,February,7781
7,2,March,9566
0,3,April,10886
8,4,May,11767
6,5,June,10878
5,6,July,12585
1,7,August,13809
11,8,September,10463
10,9,October,11093


In [263]:
fig = px.histogram(
    mon,
    x='arrival_date_month',
    y='hotel',
    title='Популярность бронирований по месяцам')

fig.update_layout(
    xaxis_title='месяц',
    yaxis_title='количество бронирований'
)
fig.show()

In [264]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

In [265]:
fig = px.line(
    mon,
    x='arrival_date_month',
    y='hotel',
    title='Популярность бронирований по месяцам',
    category_orders={'arrival_date_month': mon['arrival_date_month'].tolist()}
)

fig.update_traces(line=dict(shape='spline', width=4, color='#205072'))

fig.update_layout(
    xaxis_title='Месяц',
    yaxis_title='Количество бронирований'
)
fig.show()

Заметно, что после стабильного роста от начала года до мая тренд сначала значительно проседает в июне, а потом резко подскакивает к августу. Ближе к концу года линия тренда становится нисходящей, немного возрастая в октябре.

In [266]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

## `market_segment`

In [267]:
df.market_segment.unique()

['Direct', 'Corporate', 'Online TA', 'Offline TA/TO', 'Complementary', 'Groups', 'Aviation']
Categories (7, object): ['Aviation', 'Complementary', 'Corporate', 'Direct', 'Groups',
                         'Offline TA/TO', 'Online TA']

In [268]:
ms = df.groupby('market_segment')['hotel'].count().reset_index()

/tmp/ipykernel_2167/842779170.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [269]:
ms

,market_segment,hotel
0,Aviation,237
1,Complementary,728
2,Corporate,5096
3,Direct,12377
4,Groups,19007
5,Offline TA/TO,23902
6,Online TA,56386


Если график снизу неправильно отображается, уменьшите масштаб страницы.

In [270]:
leg = {
    'Aviation': 'для авиасотрудников',
    'Complementary': 'бесплатная',
    'Corporate': 'командировочная',
    'Direct': 'через отель напрямую',
    'Groups': 'групповое (много номеров)',
    'Offline TA/TO': 'офлайн через туроператора',
    'Online TA': 'онлайн через туроператора'
}
x = leg.values()

fig = px.bar(
    ms,
    y='hotel',
    x = x,
    title = 'Популярность видов бронирований')

fig.update_layout(
    xaxis_title='как забронирован',
    yaxis_title='количество бронирований'
)

fig.show()

Логично, что самым популярным способом бронирования отелей стал туроператор (онлайн или офлайн). Через отель напрямую бронирует номера очень мало людей.

In [271]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

## `distribution_channel`

In [272]:
df.distribution_channel.unique()

['Direct', 'Corporate', 'TA/TO', 'Undefined', 'GDS']
Categories (5, object): ['Corporate', 'Direct', 'GDS', 'TA/TO', 'Undefined']

In [273]:
df = df[df.distribution_channel != 'Undefined'].copy()
df['distribution_channel'] = df['distribution_channel'].astype('category')
df['distribution_channel'] = df['distribution_channel'].cat.remove_unused_categories()

In [274]:
df.distribution_channel.unique()

['Direct', 'Corporate', 'TA/TO', 'GDS']
Categories (4, object): ['Corporate', 'Direct', 'GDS', 'TA/TO']

In [275]:
distch = df.groupby('distribution_channel')['hotel'].count().reset_index()

/tmp/ipykernel_2167/4122291981.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [276]:
distch

,distribution_channel,hotel
0,Corporate,6444
1,Direct,14214
2,GDS,193
3,TA/TO,96881


In [277]:
leg = {
    'Corporate': 'командировочная',
    'Direct': 'через отель напрямую',
    'GDS': 'на B2B-портале',
    'TA/TO': 'через туроператора'
}

x = leg.values()

fig = px.pie(
    distch,
    names=x,
    values= distch.hotel,
    title = 'Популярность видов бронирований')

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

По этому графику выводы аналогичные.

In [278]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

## `reserved_room_type`

In [279]:
df.reserved_room_type.unique()

['C', 'A', 'D', 'E', 'G', 'F', 'H', 'L', 'B', 'P']
Categories (10, object): ['A', 'B', 'C', 'D', ..., 'G', 'H', 'L', 'P']

In [280]:
rt = df.groupby('reserved_room_type')['hotel'].count().reset_index()
rt = rt.sort_values('reserved_room_type')

/tmp/ipykernel_2167/949560664.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [281]:
rt

,reserved_room_type,hotel
0,A,84595
1,B,1114
2,C,907
3,D,19102
4,E,6453
5,F,2879
6,G,2073
7,H,601
8,L,6
9,P,2


In [282]:
rtv = {
    'A': 'стандартный',
    'B': 'улучшенный',
    'C': 'комфорт/делюкс',
    'D': 'семейный',
    'E': 'бизнес/экзекьютив',
    'F': 'полулюкс',
    'G': 'люкс',
    'H': 'президентский люкс',
    'L': 'апартаменты',
    'P': 'промо/эконом'
}
fig = px.pie(
    rt,
    names=rtv,
    values= rt.hotel,
    title = 'Популярность брони видов номеров')

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

In [283]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

## `assigned_room_type`

In [284]:
df.assigned_room_type.unique()

['C', 'A', 'D', 'E', 'G', ..., 'B', 'H', 'L', 'K', 'P']
Length: 12
Categories (12, object): ['A', 'B', 'C', 'D', ..., 'I', 'K', 'L', 'P']

In [285]:
at = df.groupby('assigned_room_type')['hotel'].count().reset_index()
at = at.sort_values('assigned_room_type')

/tmp/ipykernel_2167/1427066157.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [286]:
at

,assigned_room_type,hotel
0,A,73039
1,B,2156
2,C,2251
3,D,25043
4,E,7665
5,F,3707
6,G,2528
7,H,706
8,I,355
9,K,279


In [287]:
atv = {
    'A': 'стандартный',
    'B': 'улучшенный',
    'C': 'комфорт/делюкс',
    'D': 'семейный',
    'E': 'бизнес/экзекьютив',
    'F': 'полулюкс',
    'G': 'люкс',
    'H': 'президентский люкс',
    'I': 'смежный',
    'K': 'с кухней',
    'L': 'апартаменты',
    'P': 'промо/эконом'
}

fig = px.pie(
    at,
    names=atv,
    values= at.hotel,
    title = 'Популярность назначенных видов номеров',
    width=1500,
    height=950)


fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

Заметно, что распределение назначенных номеров существенно отличается от тех номеров, которые бронировались изначально. Существенно уменьшилась доля стандартных номеров, значительно увеличилась доля семейных номеров и немного - доля всех остальных типов.

In [288]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

## `deposit_type`

In [289]:
df.deposit_type.unique()

['No Deposit', 'Refundable', 'Non Refund']
Categories (3, object): ['No Deposit', 'Non Refund', 'Refundable']

In [290]:
dt = df.groupby('deposit_type')['hotel'].count().reset_index()
dt = dt.sort_values('deposit_type')

/tmp/ipykernel_2167/3853387325.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [291]:
dt

,deposit_type,hotel
0,No Deposit,103145
1,Non Refund,14425
2,Refundable,162


In [292]:
dtv = {
    'No Deposit': 'без предоплаты',
    'Non Refund': 'невозвратный',
    'Refundable': 'возвратный',
}

fig = px.pie(
    dt,
    names=dtv,
    values= dt.hotel,
    title = 'Популярность типов предоплаты'
)

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

In [293]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

## `customer_type`

In [294]:
df.customer_type.unique()

['Transient', 'Contract', 'Transient-Party', 'Group']
Categories (4, object): ['Contract', 'Group', 'Transient', 'Transient-Party']

In [295]:
ct = df.groupby('customer_type')['hotel'].count().reset_index()
ct = ct.sort_values('customer_type')

/tmp/ipykernel_2167/3760855327.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [296]:
ct

,customer_type,hotel
0,Contract,4061
1,Group,568
2,Transient,88789
3,Transient-Party,24314


In [297]:
ctv = {
    'Contract': 'договорные/контрактные',
    'Group': 'групповые',
    'Transient': 'индивидуальные (независимые)',
    'Transient-Party': 'связанные индивидуальные'
}

fig = px.pie(
    ct,
    names=ctv,
    values= ct.hotel,
    title = 'Распределение видов клиентов'
)

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

In [298]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

## `reservation_status`

In [299]:
df.reservation_status.unique()

['Check-Out', 'Canceled', 'No-Show']
Categories (3, object): ['Canceled', 'Check-Out', 'No-Show']

In [300]:
rt = df.groupby('reservation_status')['hotel'].count().reset_index()
rt = rt.sort_values('reservation_status')

/tmp/ipykernel_2167/2635271030.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [301]:
rt

,reservation_status,hotel
0,Canceled,42666
1,Check-Out,73865
2,No-Show,1201


In [302]:
rtv = {
    'Canceled': 'отменено',
    'Check-Out': 'успешный выезд',
    'No-Show	': 'неявка',
}

fig = px.pie(
    rt,
    names=rtv,
    values= rt.hotel,
    title = 'Распределение итоговых статусов бронирования'
)

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

Удивительно, что в этом датасете больше трети всех броней было отменено, лишь 62% бронирующих провели в отеле о

In [303]:
fig.write_html(f"{path}/{fig.layout.title.text}.html")

# Univariate analysis категориальных переменных (без дубликатов)

## `hotel`

In [304]:
fig = px.histogram(df_no_dup,
             x='hotel',
             title='Типы отелей в датасете')

fig.update_layout(
    xaxis_title='тип отеля',
    yaxis_title='количество'
)
fig.show()

In [305]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

## `meal`

In [306]:
meals = df_no_dup.groupby('meal')['hotel'].count().reset_index()

/tmp/ipykernel_2167/689541072.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [307]:
meals

,meal,hotel
0,BB,67540
1,FB,359
2,HB,9054
3,SC,9473
4,Undefined,488


В столбце есть значения 'undefined'. Избавимся от них, обновив категории.

In [308]:
df_no_dup = df_no_dup[df_no_dup.meal != 'Undefined'].copy()

In [309]:
df_no_dup['meal'] = df_no_dup['meal'].astype('category')

In [310]:
df_no_dup['meal'] = df_no_dup['meal'].cat.remove_unused_categories()

In [311]:
meals = df_no_dup.groupby('meal')['hotel'].count().reset_index()

/tmp/ipykernel_2167/689541072.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [312]:
meals

,meal,hotel
0,BB,67540
1,FB,359
2,HB,9054
3,SC,9473


In [313]:
fig = px.pie(meals,
             values='hotel',
             names='meal',
             title='Типы пансионов',
             labels={
                'BB': 'Завтрак',
                'HB': 'Полупансион',
                'FB': 'Полный пансион',
                'SC': 'Без питания'
            })

leg = {
    'BB': 'Завтрак',
    'HB': 'Полупансион',
    'FB': 'Полный пансион',
    'SC': 'Без питания'
}

fig.for_each_trace(lambda t: t.update(labels=[leg.get(x, x) for x in t.labels]))

fig.show()

Интересно, что в питание у большинства бронирующих был включен только завтрак, вовсе без питания тариф выбрали практически 10 процентов.

In [314]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

## `country`

In [315]:
df_no_dup.country.unique()

['PRT', 'GBR', 'USA', 'ESP', 'IRL', ..., 'KIR', 'SDN', 'ATF', 'SLE', 'LAO']
Length: 177
Categories (177, object): ['ABW', 'AGO', 'AIA', 'ALB', ..., 'VNM', 'ZAF', 'ZMB', 'ZWE']

Преобразуем коды стран в полное название, а также переведем названия стран на русский для более удобного восприятия.

In [316]:
import pycountry
import gettext

In [317]:
def get_country_name(code):
    try:
        country = pycountry.countries.get(alpha_3=code)
        return country.name
    except:
        return code

In [318]:
df_no_dup['country_full'] = df_no_dup['country'].astype(str).apply(get_country_name)
df_no_dup['country_full'] = df_no_dup['country_full'].astype('category')

In [319]:
countries = df_no_dup.groupby('country_full')['country'].count().reset_index()

/tmp/ipykernel_2167/319836060.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [320]:
countries

,country_full,country
0,Albania,11
1,Algeria,82
2,American Samoa,1
3,Andorra,7
4,Angola,342
...,...,...
172,"Venezuela, Bolivarian Republic of",21
173,Viet Nam,8
174,"Virgin Islands, British",1
175,Zambia,2


In [321]:
countries = countries.sort_values(by='country', ascending=False)

In [322]:
countries

,country_full,country
130,Portugal,27169
167,United Kingdom,10413
55,France,8768
149,Spain,7160
60,Germany,5384
...,...,...
116,New Caledonia,1
118,Nicaragua,1
151,Sudan,1
169,United States Minor Outlying Islands,1


In [323]:
countries['log_count'] = np.log(countries['country'])

In [324]:
fig = px.choropleth(
    countries,
    locations="country_full",
    locationmode="country names",
    color="log_count",
    color_continuous_scale = "Tealrose",
    title="География гостей в отелях (логнормированная)",
    hover_data={
        "log_count": False,
        "country_full": True,
        "country": True
    },
    labels={
        "country_full": "Страна",
        "country": "Количество гостей",
        "log_count": "логарифм количества гостей"
    }
)

fig.show()

Больше всего броней исходило от жителей стран Западной Европы, а также Великобритании, США, Бразилии. Меньше всего - от жителей из стран Африки, Юго-Восточной Азии и Океании.

In [325]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

## `arrival_date_month`

Чтобы правильно отсортировать данные на графике, добавлю столбец с номером месяца.

In [326]:
mlist = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
df_no_dup['arrival_date_month_num'] = df_no_dup['arrival_date_month'].apply(lambda x: mlist.index(x))

In [327]:
mon = df_no_dup.groupby(['arrival_date_month_num', 'arrival_date_month'], observed=True)['hotel'].count().reset_index()

In [328]:
mon['arrival_date_month_num'] = mon['arrival_date_month_num'].astype(int)
mon = mon.sort_values('arrival_date_month_num')

In [329]:
mon

,arrival_date_month_num,arrival_date_month,hotel
4,0,January,4574
3,1,February,5978
7,2,March,7400
0,3,April,7805
8,4,May,8335
6,5,June,7731
5,6,July,9979
1,7,August,11187
11,8,September,6653
10,9,October,6881


In [330]:
fig = px.histogram(
    mon,
    x='arrival_date_month',
    y='hotel',
    title='Популярность бронирований по месяцам')

fig.update_layout(
    xaxis_title='месяц',
    yaxis_title='количество бронирований'
)
fig.show()

In [331]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

In [332]:
fig = px.line(
    mon,
    x='arrival_date_month',
    y='hotel',
    title='Популярность бронирований по месяцам',
    category_orders={'arrival_date_month': mon['arrival_date_month'].tolist()}
)

fig.update_traces(line=dict(shape='spline', width=4, color='#205072'))

fig.update_layout(
    xaxis_title='Месяц',
    yaxis_title='Количество бронирований'
)
fig.show()

Заметно, что после стабильного роста от начала года до мая тренд сначала значительно проседает в июне, а потом резко подскакивает к августу. Ближе к концу года линия тренда становится нисходящей, немного возрастая в октябре.

In [333]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

## `market_segment`

In [334]:
df_no_dup.market_segment.unique()

['Direct', 'Corporate', 'Online TA', 'Offline TA/TO', 'Complementary', 'Groups', 'Aviation']
Categories (7, object): ['Aviation', 'Complementary', 'Corporate', 'Direct', 'Groups',
                         'Offline TA/TO', 'Online TA']

In [335]:
ms = df_no_dup.groupby('market_segment')['hotel'].count().reset_index()

/tmp/ipykernel_2167/529146716.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [336]:
ms

,market_segment,hotel
0,Aviation,227
1,Complementary,692
2,Corporate,4020
3,Direct,11575
4,Groups,4711
5,Offline TA/TO,13683
6,Online TA,51518


Если график снизу неправильно отображается, уменьшите масштаб страницы.

In [337]:
leg = {
    'Aviation': 'для авиасотрудников',
    'Complementary': 'бесплатная',
    'Corporate': 'командировочная',
    'Direct': 'через отель напрямую',
    'Groups': 'групповое (много номеров)',
    'Offline TA/TO': 'офлайн через туроператора',
    'Online TA': 'онлайн через туроператора'
}
x = leg.values()

fig = px.bar(
    ms,
    y='hotel',
    x = x,
    title = 'Популярность видов бронирований')

fig.update_layout(
    xaxis_title='как забронирован',
    yaxis_title='количество бронирований'
)

fig.show()

Логично, что самым популярным способом бронирования отелей стал туроператор (онлайн или офлайн). Через отель напрямую бронирует номера очень мало людей.

In [338]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

## `distribution_channel`

In [339]:
df_no_dup.distribution_channel.unique()

['Direct', 'Corporate', 'TA/TO', 'Undefined', 'GDS']
Categories (5, object): ['Corporate', 'Direct', 'GDS', 'TA/TO', 'Undefined']

In [340]:
df_no_dup = df_no_dup[df_no_dup.distribution_channel != 'Undefined'].copy()
df_no_dup['distribution_channel'] = df_no_dup['distribution_channel'].astype('category')
df_no_dup['distribution_channel'] = df_no_dup['distribution_channel'].cat.remove_unused_categories()

In [341]:
df_no_dup.distribution_channel.unique()

['Direct', 'Corporate', 'TA/TO', 'GDS']
Categories (4, object): ['Corporate', 'Direct', 'GDS', 'TA/TO']

In [342]:
distch = df_no_dup.groupby('distribution_channel')['hotel'].count().reset_index()

/tmp/ipykernel_2167/100714243.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [343]:
distch

,distribution_channel,hotel
0,Corporate,4882
1,Direct,12675
2,GDS,181
3,TA/TO,68687


In [344]:
leg = {
    'Corporate': 'командировочная',
    'Direct': 'через отель напрямую',
    'GDS': 'на B2B-портале',
    'TA/TO': 'через туроператора'
}

x = leg.values()

fig = px.pie(
    distch,
    names=x,
    values= distch.hotel,
    title = 'Популярность видов бронирований')

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

По этому графику выводы аналогичные.

In [345]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

## `reserved_room_type`

In [346]:
df_no_dup.reserved_room_type.unique()

['C', 'A', 'D', 'E', 'G', 'F', 'H', 'L', 'B', 'P']
Categories (10, object): ['A', 'B', 'C', 'D', ..., 'G', 'H', 'L', 'P']

In [347]:
rt = df_no_dup.groupby('reserved_room_type')['hotel'].count().reset_index()
rt = rt.sort_values('reserved_room_type')

/tmp/ipykernel_2167/645233306.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [348]:
rt

,reserved_room_type,hotel
0,A,55806
1,B,995
2,C,890
3,D,17318
4,E,5977
5,F,2805
6,G,2031
7,H,596
8,L,6
9,P,1


In [349]:
rtv = {
    'A': 'стандартный',
    'B': 'улучшенный',
    'C': 'комфорт/делюкс',
    'D': 'семейный',
    'E': 'бизнес/экзекьютив',
    'F': 'полулюкс',
    'G': 'люкс',
    'H': 'президентский люкс',
    'L': 'апартаменты',
    'P': 'промо/эконом'
}
fig = px.pie(
    rt,
    names=rtv,
    values= rt.hotel,
    title = 'Популярность брони видов номеров')

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)
fig.update_layout(height=800)
fig.show()

In [350]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

## `assigned_room_type`

In [351]:
df_no_dup.assigned_room_type.unique()

['C', 'A', 'D', 'E', 'G', ..., 'B', 'H', 'L', 'K', 'P']
Length: 12
Categories (12, object): ['A', 'B', 'C', 'D', ..., 'I', 'K', 'L', 'P']

In [352]:
at = df_no_dup.groupby('assigned_room_type')['hotel'].count().reset_index()
at = at.sort_values('assigned_room_type')

/tmp/ipykernel_2167/1039611508.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [353]:
at

,assigned_room_type,hotel
0,A,45885
1,B,1813
2,C,2066
3,D,22198
4,E,7076
5,F,3587
6,G,2473
7,H,700
8,I,349
9,K,276


In [354]:
atv = {
    'A': 'стандартный',
    'B': 'улучшенный',
    'C': 'комфорт/делюкс',
    'D': 'семейный',
    'E': 'бизнес/экзекьютив',
    'F': 'полулюкс',
    'G': 'люкс',
    'H': 'президентский люкс',
    'I': 'смежный',
    'K': 'с кухней',
    'L': 'апартаменты',
    'P': 'промо/эконом'
}

fig = px.pie(
    at,
    names=atv,
    values= at.hotel,
    title = 'Популярность назначенных видов номеров')


fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)
fig.update_layout(height=1000)
fig.update_traces(
    textposition='inside',
    textinfo='label+percent',
    insidetextorientation='radial'
)

fig.show()

Заметно, что распределение назначенных номеров существенно отличается от тех номеров, которые бронировались изначально. Существенно уменьшилась доля стандартных номеров, значительно увеличилась доля семейных номеров и немного - доля всех остальных типов.

In [355]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

## `deposit_type`

In [356]:
df_no_dup.deposit_type.unique()

['No Deposit', 'Refundable', 'Non Refund']
Categories (3, object): ['No Deposit', 'Non Refund', 'Refundable']

In [357]:
dt = df_no_dup.groupby('deposit_type')['hotel'].count().reset_index()
dt = dt.sort_values('deposit_type')

/tmp/ipykernel_2167/448548073.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [358]:
dt

,deposit_type,hotel
0,No Deposit,85288
1,Non Refund,1030
2,Refundable,107


In [359]:
dtv = {
    'No Deposit': 'без предоплаты',
    'Non Refund': 'невозвратный',
    'Refundable': 'возвратный',
}

fig = px.pie(
    dt,
    names=dtv,
    values= dt.hotel,
    title = 'Популярность типов предоплаты'
)

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

In [360]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

## `customer_type`

In [361]:
df_no_dup.customer_type.unique()

['Transient', 'Contract', 'Transient-Party', 'Group']
Categories (4, object): ['Contract', 'Group', 'Transient', 'Transient-Party']

In [362]:
ct = df_no_dup.groupby('customer_type')['hotel'].count().reset_index()
ct = ct.sort_values('customer_type')

/tmp/ipykernel_2167/860635723.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [363]:
ct

,customer_type,hotel
0,Contract,3125
1,Group,535
2,Transient,71337
3,Transient-Party,11428


In [364]:
ctv = {
    'Contract': 'договорные/контрактные',
    'Group': 'групповые',
    'Transient': 'индивидуальные (независимые)',
    'Transient-Party': 'связанные индивидуальные'
}

fig = px.pie(
    ct,
    names=ctv,
    values= ct.hotel,
    title = 'Распределение видов клиентов'
)

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

In [365]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

## `reservation_status`

In [366]:
df_no_dup.reservation_status.unique()

['Check-Out', 'Canceled', 'No-Show']
Categories (3, object): ['Canceled', 'Check-Out', 'No-Show']

In [367]:
rt = df_no_dup.groupby('reservation_status')['hotel'].count().reset_index()
rt = rt.sort_values('reservation_status')

/tmp/ipykernel_2167/3741174719.py:1: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



In [368]:
rt

,reservation_status,hotel
0,Canceled,22893
1,Check-Out,62524
2,No-Show,1008


In [369]:
rtv = {
    'Canceled': 'отменено',
    'Check-Out': 'успешный выезд',
    'No-Show	': 'неявка',
}

fig = px.pie(
    rt,
    names=rtv,
    values= rt.hotel,
    title = 'Распределение итоговых статусов бронирования'
)

fig.update_layout(
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.1)
)

fig.show()

Удивительно, что в этом датасете больше трети всех броней было отменено, лишь 62% бронирующих провели в отеле о

In [370]:
fig.write_html(f"{path}/{fig.layout.title.text} без дубликатов.html")

# Сохранение измененного файла

In [371]:
df.to_parquet('main_df_edited.parquet')
df_no_dup.to_parquet('main_df_edited_no_dup.parquet')

In [372]:
# #если возникают проблемы
# from google.colab import drive
# drive.mount('/content/drive')

# df.to_parquet('/content/drive/MyDrive/gp_iad/main_df_edited.parquet', index=False)
# df_no_dup.to_parquet('/content/drive/MyDrive/gp_iad/main_df_edited_no_dup.parquet', index=False)